## Imports

In [40]:
import os
import pandas as pd
import random
from PIL import Image
from collections import Counter

In [37]:
data_dir = "data"

data = []

for folder_name in os.listdir(data_dir):
    folder_path = os.path.join(data_dir, folder_name)
    if os.path.isdir(folder_path):
        for file_name in os.listdir(folder_path):
            data.append({"path": str(folder_name + '_' + file_name), "class": folder_name.lower()})

raw_df = pd.DataFrame(data)

In [39]:
print(raw_df.describe())
print(raw_df.info())

print(f"Classes: {raw_df['class'].unique()}")
print(raw_df['class'].value_counts())

                       path   class
count                  4793    4793
unique                 4793       5
top     Moped_PHOTO_510.jpg  pickup
freq                      1    1689
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4793 entries, 0 to 4792
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   path    4793 non-null   object
 1   class   4793 non-null   object
dtypes: object(2)
memory usage: 75.0+ KB
None
Classes: ['moped' 'hatchback' 'pickup' 'suv' 'seden']
class
pickup       1689
seden        1222
suv           680
hatchback     602
moped         600
Name: count, dtype: int64


# Preprocessing

### Steps:
- Data Resizing
- Data Augmentation
- Data Normalisation

## Data Resizing

Investigating the current sizes of the images and resizing as needed, to a minimum of 30x30x3 dimensions

In [29]:
def check_image_sizes(data_dir):
    image_sizes = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    image_sizes.append(img.size)
            except Exception as e:
                print(f"Error openeing {file_path}: {e}")

    size_distributions = Counter(image_sizes)
    print("Image Size Distribution:")
    for size, count in size_distributions.items():
        print(f"{size}: {count} images")

check_image_sizes("./data")

Image Size Distribution:
(462, 739): 1 images
(324, 493): 1 images
(159, 290): 1 images
(221, 368): 1 images
(274, 185): 1 images
(401, 430): 2 images
(320, 560): 1 images
(346, 524): 1 images
(412, 659): 2 images
(107, 128): 1 images
(253, 428): 1 images
(377, 458): 2 images
(454, 746): 2 images
(330, 458): 1 images
(498, 607): 4 images
(228, 348): 2 images
(234, 410): 2 images
(240, 400): 2 images
(252, 469): 2 images
(280, 435): 2 images
(158, 282): 2 images
(95, 117): 1 images
(336, 485): 2 images
(90, 134): 1 images
(380, 418): 1 images
(138, 169): 1 images
(389, 707): 2 images
(244, 352): 1 images
(489, 769): 1 images
(423, 671): 1 images
(156, 127): 2 images
(56, 110): 1 images
(317, 573): 2 images
(391, 403): 2 images
(324, 536): 1 images
(270, 456): 1 images
(340, 581): 2 images
(231, 438): 1 images
(412, 470): 1 images
(153, 287): 1 images
(383, 627): 1 images
(462, 763): 1 images
(125, 130): 1 images
(241, 453): 1 images
(393, 429): 1 images
(187, 342): 2 images
(334, 460): 

We can see most of the images are different sizes, so we must transform them into a uniform size.

In [25]:
def resize_image(input_path: str, output_path: str, size: tuple = (224, 224)):
    """
    Resizes an image to the specified size and saves it to output path.

    Args:
        input_path (str): Path to the input image.
        output_path (str): Path to save the resized image.
        size (tuple(int, int)): The dimensions to resize the image to.
    
    Raises:
        Exception when error occurs resizing.
    """
    try:
        with Image.open(input_path) as img:
            img = img.resize(size, Image.Resampling.LANCZOS)
            img.save(output_path)
    except Exception as e:
        print(f"Error resizing {input_path}: {e}")

In [26]:
# Directory paths
input_dir = "./data"  # Original dataset
output_dir = "./uniform_data"  # Resized dataset
os.makedirs(output_dir, exist_ok=True)

# Resize all images
for root, dirs, files in os.walk(input_dir):
    for file in files:
        input_path = os.path.join(root, file)
        # Create corresponding output path
        relative_path = os.path.relpath(root, input_dir)
        output_class_dir = os.path.join(output_dir, relative_path)
        os.makedirs(output_class_dir, exist_ok=True)
        output_path = os.path.join(output_class_dir, file)

        # Resize image
        resize_image(input_path, output_path, size=(224, 224))

print("All images resized and saved to", output_dir)

All images resized and saved to ./uniform_data


In [30]:
check_image_sizes("./uniform_data")

Image Size Distribution:
(224, 224): 4793 images


Now all images are resized to uniform, we do a sanity check for 3 channels

In [ ]:
def verify_image_modes(directory):
    modes = []
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with Image.open(file_path) as img:
                    modes.append(img.mode)
            except Exception as e:
                print(f"Error opening {file_path}: {e}")
    return set(modes)

modes = verify_image_modes("./uniform_data")
print("Image Modes:", modes) 

Image Modes: {'RGB'}


## Augmentation
Augmentation includes zooming, flipping, or shifting images. This increases data diversity for the model to learn from. However I want to make sure there is still clear pictures for the model to use, so augmentation will be applied to only 15% of the images.

In [ ]:
def augment_images(df, datagen, output_dir, amount=0.15):
    for class_name in df['class'].unique():
        class_df = df[df['class'] == class_name]
        class_images = class_df['path'].to_list()

        num_to_augment = max(1, int(len(class_images) * amount))
        selected_images = random.sample(class_images, num_to_augment)

        class_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        for input_path in selected_images:
            with Image.open(input_path) as img:
                
                for i, augmented_images in enumerate(datagen.flow(img_array))